# RQ2: Feature Group Effectiveness

**Research question:** How does the use of different feature groups, including content-related, platform-related, temporal, and creator-related features, affect prediction performance?

This notebook uses the cleaned train/validation/test splits created by `notebooks/preprocessing.ipynb`. It uses the original processed dataset rather than the ML-ready CSV because the original feature columns are more transparent and less likely to contain target-adjacent engineered signal.

The experiment has two parts:

1. **Group-only models:** train with one feature group at a time.
2. **Cumulative models:** add feature groups step by step and measure whether validation performance improves.

Engagement features are included because they are part of the dataset and relevant to RQ2, but they should be interpreted as post-publication explanatory variables rather than features available before a video is uploaded.

## 1. Experiment Setup

In [ ]:
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.api.types import is_bool_dtype, is_object_dtype, is_string_dtype
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "trend_label"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

## 2. Load Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)
test = pd.read_csv(TEST_PATH)

X_train = train.drop(columns=TARGET)
y_train = train[TARGET]
X_val = validation.drop(columns=TARGET)
y_val = validation[TARGET]
X_test = test.drop(columns=TARGET)
y_test = test[TARGET]

split_summary = pd.DataFrame([
    {"split": "train", "rows": len(train), "columns": train.shape[1]},
    {"split": "validation", "rows": len(validation), "columns": validation.shape[1]},
    {"split": "test", "rows": len(test), "columns": test.shape[1]},
])

display(split_summary)
display(pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "validation": y_val.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).round(4))

The target classes are close to balanced, so macro-F1 is the main comparison metric. Accuracy is reported as supporting context.

## 3. Define Feature Groups

In [ ]:
feature_groups = {
    "platform_region": [
        "platform", "country", "region", "language", "device_type", "device_brand", "traffic_source",
    ],
    "content": [
        "category", "genre", "hashtag", "sound_type", "duration_sec", "title_length", "has_emoji",
    ],
    "temporal": [
        "week_of_year", "upload_hour", "publish_dayofweek", "publish_period", "event_season",
        "season", "is_weekend", "publish_month", "publish_day",
    ],
    "creator": ["creator_avg_views", "creator_tier"],
    "engagement": [
        "views", "likes", "comments", "shares", "saves", "engagement_rate", "dislikes",
        "comment_ratio", "share_rate", "save_rate", "like_dislike_ratio", "engagement_total",
        "like_rate", "dislike_rate", "engagement_per_1k", "engagement_like_rate",
        "engagement_comment_rate", "engagement_share_rate", "completion_rate", "avg_watch_time_sec",
    ],
}

feature_groups = {
    group: [column for column in columns if column in X_train.columns]
    for group, columns in feature_groups.items()
}

group_summary = pd.DataFrame([
    {"feature_group": group, "n_columns": len(columns), "columns": ", ".join(columns)}
    for group, columns in feature_groups.items()
])

display(group_summary[["feature_group", "n_columns"]])

covered_columns = sorted({column for columns in feature_groups.values() for column in columns})
uncovered_columns = sorted(set(X_train.columns) - set(covered_columns))
print("Total grouped columns:", len(covered_columns))
print("Ungrouped columns:", uncovered_columns)

The groups cover all 45 predictors in the processed dataset. The non-engagement groups represent features that are closer to metadata available at or near upload time. The engagement group represents observed audience response.

## 4. Modelling Helpers

In [ ]:
def is_categorical(series: pd.Series) -> bool:
    return (
        is_object_dtype(series)
        or is_string_dtype(series)
        or is_bool_dtype(series)
        or isinstance(series.dtype, pd.CategoricalDtype)
    )


def split_column_types(X: pd.DataFrame):
    categorical_columns = [column for column in X.columns if is_categorical(X[column])]
    numeric_columns = [column for column in X.columns if column not in categorical_columns]
    return categorical_columns, numeric_columns


def make_preprocessor(X: pd.DataFrame, scale_numeric: bool):
    categorical_columns, numeric_columns = split_column_types(X)

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False)),
    ])

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_transformer = Pipeline(steps=numeric_steps)

    return ColumnTransformer(
        transformers=[
            ("cat", categorical_transformer, categorical_columns),
            ("num", numeric_transformer, numeric_columns),
        ],
        verbose_feature_names_out=False,
    )


def make_logistic_pipeline(X: pd.DataFrame):
    return Pipeline(steps=[
        ("preprocess", make_preprocessor(X, scale_numeric=True)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            C=0.5,
            random_state=RANDOM_STATE,
        )),
    ])


def make_random_forest_pipeline(X: pd.DataFrame):
    return Pipeline(steps=[
        ("preprocess", make_preprocessor(X, scale_numeric=False)),
        ("model", RandomForestClassifier(
            n_estimators=100,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


def evaluate_predictions(y_true, y_pred) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def fit_and_evaluate(model_name: str, model, feature_set_name: str, columns: list[str]) -> dict:
    start = time.time()
    fitted_model = clone(model).fit(X_train[columns], y_train)
    predictions = fitted_model.predict(X_val[columns])
    metrics = evaluate_predictions(y_val, predictions)
    metrics.update({
        "model": model_name,
        "feature_set": feature_set_name,
        "n_columns": len(columns),
        "fit_seconds": time.time() - start,
    })
    return metrics

Two models are used here: a regularized linear model and a random forest. RQ3 handles broader model-family comparison, so RQ2 keeps the modelling choices limited and focuses on feature groups.

## 5. Baseline

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_val)

baseline_metrics = evaluate_predictions(y_val, baseline_predictions)
baseline_metrics.update({
    "model": "majority_baseline",
    "feature_set": "none",
    "n_columns": 0,
    "fit_seconds": 0.0,
})

display(pd.DataFrame([baseline_metrics]).round(4))

Because the classes are balanced, the majority baseline has accuracy close to 25% but very low macro-F1. A useful feature group should improve macro-F1 meaningfully above this baseline and ideally above the chance-level region around 0.25.

## 6. Group-only Experiments

In [ ]:
models = {
    "logistic_regression": make_logistic_pipeline,
    "random_forest": make_random_forest_pipeline,
}

group_only_rows = [baseline_metrics]

for feature_set_name, columns in feature_groups.items():
    for model_name, model_factory in models.items():
        model = model_factory(X_train[columns])
        row = fit_and_evaluate(model_name, model, feature_set_name, columns)
        group_only_rows.append(row)

group_only_results = pd.DataFrame(group_only_rows)
group_only_results = group_only_results[
    ["model", "feature_set", "n_columns", "accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall", "fit_seconds"]
].sort_values("macro_f1", ascending=False)

display(group_only_results.round(4))

In [ ]:
plot_df = group_only_results[group_only_results["model"] != "majority_baseline"].copy()
plot_df = plot_df.sort_values(["model", "macro_f1"])

fig, ax = plt.subplots(figsize=(9, 5))
for model_name, model_df in plot_df.groupby("model"):
    ax.plot(model_df["macro_f1"], model_df["feature_set"], marker="o", label=model_name)

ax.axvline(baseline_metrics["macro_f1"], color="black", linestyle="--", linewidth=1, label="majority baseline")
ax.set_xlabel("Validation macro-F1")
ax.set_ylabel("Feature group")
ax.set_title("Group-only validation performance")
ax.legend()
plt.tight_layout()

## 7. Cumulative Feature Addition

In [ ]:
def unique_columns(groups: list[str]) -> list[str]:
    columns = []
    seen = set()
    for group in groups:
        for column in feature_groups[group]:
            if column not in seen:
                columns.append(column)
                seen.add(column)
    return columns


cumulative_feature_sets = {
    "platform_region": unique_columns(["platform_region"]),
    "platform_region_content": unique_columns(["platform_region", "content"]),
    "plus_temporal": unique_columns(["platform_region", "content", "temporal"]),
    "non_engagement_all": unique_columns(["platform_region", "content", "temporal", "creator"]),
    "all_features": unique_columns(["platform_region", "content", "temporal", "creator", "engagement"]),
}

display(pd.DataFrame([
    {"feature_set": name, "n_columns": len(columns)}
    for name, columns in cumulative_feature_sets.items()
]))

In [ ]:
cumulative_rows = [baseline_metrics]

for feature_set_name, columns in cumulative_feature_sets.items():
    for model_name, model_factory in models.items():
        model = model_factory(X_train[columns])
        row = fit_and_evaluate(model_name, model, feature_set_name, columns)
        cumulative_rows.append(row)

cumulative_results = pd.DataFrame(cumulative_rows)
cumulative_results = cumulative_results[
    ["model", "feature_set", "n_columns", "accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall", "fit_seconds"]
].sort_values(["model", "macro_f1"], ascending=[True, False])

display(cumulative_results.round(4))

In [ ]:
ordered_sets = list(cumulative_feature_sets.keys())
plot_df = cumulative_results[cumulative_results["model"].isin(models.keys())].copy()
plot_df["feature_set"] = pd.Categorical(plot_df["feature_set"], categories=ordered_sets, ordered=True)
plot_df = plot_df.sort_values("feature_set")

fig, ax = plt.subplots(figsize=(10, 5))
for model_name, model_df in plot_df.groupby("model"):
    ax.plot(model_df["feature_set"].astype(str), model_df["macro_f1"], marker="o", label=model_name)

ax.axhline(baseline_metrics["macro_f1"], color="black", linestyle="--", linewidth=1, label="majority baseline")
ax.set_xlabel("Cumulative feature set")
ax.set_ylabel("Validation macro-F1")
ax.set_title("Effect of adding feature groups")
ax.tick_params(axis="x", rotation=25)
ax.legend()
plt.tight_layout()

## 8. Validation-to-Test Check

The best validation feature set for each model is evaluated once on the held-out test set. This is not intended to tune the model further; it checks whether the validation pattern survives on unseen data.

In [ ]:
combined_train_validation = pd.concat([train, validation], ignore_index=True)
X_train_val = combined_train_validation.drop(columns=TARGET)
y_train_val = combined_train_validation[TARGET]

best_validation_rows = (
    cumulative_results[cumulative_results["model"].isin(models.keys())]
    .sort_values("macro_f1", ascending=False)
    .groupby("model", as_index=False)
    .head(1)
)

test_rows = []

for _, best_row in best_validation_rows.iterrows():
    model_name = best_row["model"]
    feature_set_name = best_row["feature_set"]
    columns = cumulative_feature_sets[feature_set_name]
    model = models[model_name](X_train_val[columns])
    fitted_model = model.fit(X_train_val[columns], y_train_val)
    predictions = fitted_model.predict(X_test[columns])
    metrics = evaluate_predictions(y_test, predictions)
    metrics.update({
        "model": model_name,
        "selected_feature_set": feature_set_name,
        "n_columns": len(columns),
        "validation_macro_f1": best_row["macro_f1"],
    })
    test_rows.append(metrics)

test_results = pd.DataFrame(test_rows)[[
    "model", "selected_feature_set", "n_columns", "validation_macro_f1",
    "accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall",
]]

display(test_results.round(4))

## 9. RQ2 Interpretation

The key result is that changing feature groups does **not** produce a large or stable improvement in trend-label prediction on the original processed dataset. Most validation macro-F1 scores remain close to the chance-level region around 0.25.

The strongest broad pattern is usually that **engagement features** perform as well as or slightly better than the other single groups. This makes intuitive sense because views, likes, comments, shares, saves, watch-time, and completion-rate variables describe audience response after publication. However, these features are less useful for a strict pre-upload prediction setting.

**Content metadata** and **temporal context** sometimes provide small gains, but the gains are weak and model-dependent. **Platform/region metadata** and **creator attributes** do not clearly improve validation macro-F1 by themselves.

The cumulative experiment is the most important RQ2 evidence. If feature groups were strongly complementary, macro-F1 should rise clearly as more groups are added. Instead, the curve stays almost flat. This suggests that the original transparent features contain limited predictive signal for the provided `trend_label`, or that the labels are noisy/synthetic enough that standard metadata cannot reliably separate rising, stable, declining, and seasonal videos.

For the report, this can be framed as follows: feature engineering and group selection have only marginal effects on the original dataset. Engagement variables are the most informative group, but even they do not produce strong generalisable performance. Therefore, the project should avoid claiming that any feature group strongly determines video trend class.